# CA1 — Building a Categorical Naive Bayes Classifier

The target columns are `poisonous` for Mushroom Dataset and `class` for Soybean Dataset.

## Part 3

In [30]:
import numpy as np
import pandas as pd

# TODO : fill each dataframe from csv files using pd.read_csv

full_test = pd.read_csv("mushroom_full_test.csv")
full_train = pd.read_csv("mushroom_full_train.csv")
small_train = pd.read_csv("mushroom_small_train.csv")
small_test = pd.read_csv("mushroom_small_test.csv")
soy_train = pd.read_csv("soybean_train.csv")
soy_test = pd.read_csv("soybean_test.csv")


In [31]:
TARGET_MUSHROOM = "poisonous"
TARGET_CLASSES = ["p", "f"]
SMALL_FEATURES = ["bruises", "gill-size"]

def model_creation(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = len(subset_class) / total_samples

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                prob = match_count / len(subset_class)
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_MUSHROOM]
    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = model["conditionals"].get((class_, feature, value), 0.0)
                total_prob *= prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)
        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

small_model_direct = model_creation(small_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=SMALL_FEATURES)
small_predictions_direct = predict(
    small_model_direct, small_test[SMALL_FEATURES]
)
print(f"Small Mushroom baseline accuracy: "
      f"{accuracy(small_test[TARGET_MUSHROOM], small_predictions_direct):.2%}")

ALL_COLUMNS = [c for c in full_test.columns if c != TARGET_MUSHROOM]
big_model_direct = model_creation(full_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, full_test[ALL_COLUMNS]
)
print(f"Full Mushroom baseline accuracy: "
      f"{accuracy(full_test[TARGET_MUSHROOM], full_predictions_direct):.2%}")


Small Mushroom baseline accuracy: 76.80%
Full Mushroom baseline accuracy: 99.45%


مورد د : 

1: خیر دقت مدل در هر دو مجموعه داده های small full یکی نیست  

2: استفاده از ویژگی های بیشتر در داده ها و فرآیند یادگیری باعث افزایش میران دقت در تخمین پیش بینی را دارد  

3: خیر نمی توان این نتیجه را گرفت به این علت مه افزایش بیش از حد ویژگی باعص می شود که مدل این الگو ها رفتاری را یاد نگیرد بلکه داده ها و موقعیت ها را حفظ کند به همین سبب داده ها و ویژگی های کلید باید اضافه شوند و سایر ویژگی ها در نظر گرفته نشوند.

## Part 4

In [32]:
TARGET_MUSHROOM = "poisonous"
TARGET_CLASSES = ["p", "f"]
SMALL_FEATURES = ["bruises", "gill-size"]

def model_creation(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = np.log(len(subset_class) / total_samples)

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                # This part is for the exception of 0 in the np.log
                if match_count == 0:
                    prob = 0
                    print("Under flow has happened!!")
                else:
                    prob = np.log(match_count / len(subset_class))
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_MUSHROOM]

    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = model["conditionals"].get((class_, feature, value), 0.0)
                total_prob += prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)
        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

small_model_direct = model_creation(small_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=SMALL_FEATURES)
small_predictions_direct = predict(
    small_model_direct, small_test[SMALL_FEATURES]
)
print(f"Small Mushroom baseline accuracy: "
      f"{accuracy(small_test[TARGET_MUSHROOM], small_predictions_direct):.2%}")

ALL_COLUMNS = [c for c in full_test.columns if c != TARGET_MUSHROOM]
big_model_direct = model_creation(full_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, full_test[ALL_COLUMNS]
)
print(f"Full Mushroom baseline accuracy: "
      f"{accuracy(full_test[TARGET_MUSHROOM], full_predictions_direct):.2%}")


Small Mushroom baseline accuracy: 76.80%
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened!!
Under flow has happened

1: خبر دقت نسخه پیشین و لگاریتمی یکی نیست  
2: با توحه به خروحی داده شده وقتی این اختمال 0 می شود خروحی Underflow has happened خررا خواهیم داشت از طرفی  وقتی این پدیده را داریم  
برخی از حالت اختمال هایی که بهشان نسبت داده می شوند 0 می شود در این صورت مدل دقت واقغ بینانه ای نسب به پیش بینی نخواهد داشت.

## Part 5

In [33]:
soy_train.columns

Index(['date', 'plant-stand', 'precip', 'temp', 'hail', 'crop-hist',
       'area-damaged', 'severity', 'seed-tmt', 'germination', 'plant-growth',
       'leaves', 'leafspots-halo', 'leafspots-marg', 'leafspot-size',
       'leaf-shread', 'leaf-malf', 'leaf-mild', 'stem', 'lodging',
       'stem-cankers', 'canker-lesion', 'fruiting-bodies', 'external-decay',
       'mycelium', 'int-discolor', 'sclerotia', 'fruit-pods', 'fruit-spots',
       'seed', 'mold-growth', 'seed-discolor', 'seed-size', 'shriveling',
       'roots', 'class'],
      dtype='str')

فرآیند آموزش بدون لگاریتم

In [41]:
TARGET_MUSHROOM = "class"
ALL_COLUMNS = [c for c in soy_train.columns if c != TARGET_MUSHROOM]


def model_creation(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = (len(subset_class) / total_samples)

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                prob = (match_count / len(subset_class))
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_MUSHROOM]

    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = model["conditionals"].get((class_, feature, value), 0.0)
                total_prob *= prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)
        if scores[best_class] == 0:
            print(f"index {_} : scores: {scores}")

        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

big_model_direct = model_creation(soy_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, soy_test[ALL_COLUMNS]
)
print(f"Full Mushroom baseline accuracy: "
      f"{accuracy(soy_test[TARGET_MUSHROOM], full_predictions_direct):.2%}")


index 0 : scores: {'2-4-d-injury': np.float64(0.0), 'alternarialeaf-spot': np.float64(0.0), 'anthracnose': np.float64(0.0), 'bacterial-blight': np.float64(0.0), 'bacterial-pustule': np.float64(0.0), 'brown-spot': np.float64(0.0), 'brown-stem-rot': np.float64(0.0), 'charcoal-rot': np.float64(0.0), 'cyst-nematode': np.float64(0.0), 'diaporthe-pod-&-stem-blight': np.float64(0.0), 'diaporthe-stem-canker': np.float64(0.0), 'downy-mildew': np.float64(0.0), 'frog-eye-leaf-spot': np.float64(0.0), 'herbicide-injury': np.float64(0.0), 'phyllosticta-leaf-spot': np.float64(0.0), 'phytophthora-rot': np.float64(0.0), 'powdery-mildew': np.float64(0.0), 'purple-seed-stain': np.float64(0.0), 'rhizoctonia-root-rot': np.float64(0.0)}
index 1 : scores: {'2-4-d-injury': np.float64(0.0), 'alternarialeaf-spot': np.float64(0.0), 'anthracnose': np.float64(0.0), 'bacterial-blight': np.float64(0.0), 'bacterial-pustule': np.float64(0.0), 'brown-spot': np.float64(0.0), 'brown-stem-rot': np.float64(0.0), 'charcoal-

آموزش داده با لگاریتم

In [50]:
TARGET_MUSHROOM = "class"
ALL_COLUMNS = [c for c in soy_train.columns if c != TARGET_MUSHROOM]


def model_creation(small_train: pd.DataFrame, TARGET_COLUMN: str, SMALL_FEATURES: list):
    model = {
        "priors": {},
        "conditionals": {}
    }
    classes = small_train[TARGET_COLUMN].unique()
    total_samples = len(small_train)

    for class_ in classes:
        subset_class = small_train[small_train[TARGET_COLUMN] == class_]
        model["priors"][class_] = np.log(len(subset_class) / total_samples)

        for feature in SMALL_FEATURES:
            unique_values = small_train[feature].unique()
            
            for value in unique_values:
                match_count = (subset_class[feature] == value).sum()
                prob = np.log(match_count / len(subset_class))
                model["conditionals"][(class_, feature, value)] = prob

    return model

def predict(model: dict, sample_frame: pd.DataFrame, TARGET_COLUMN=None):
    classes = list(model["priors"].keys())
    features = [c for c in sample_frame.columns if c != TARGET_MUSHROOM]

    predictions = []

    for _, row in sample_frame.iterrows():
        scores: dict = {}
        for class_ in classes:
            total_prob = model["priors"][class_]

            for feature in features:
                value = row[feature]
                prob = model["conditionals"].get((class_, feature, value), 1)
                if prob == 0:
                    print(f"The prob of the featue {feature} based on the calss {class_} is Zero")
                    prob = -(np.inf)
            
                total_prob += prob

            scores[class_] = total_prob

        best_class = max(scores, key=scores.get)
        if scores[best_class] == 0:
            print(f"index {_} : scores: {scores}")

        predictions.append(best_class)

    return predictions

def accuracy(y_true, y_pred):
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred)) / len(y_true)

big_model_direct = model_creation(soy_train, TARGET_COLUMN=TARGET_MUSHROOM, SMALL_FEATURES=ALL_COLUMNS)
full_predictions_direct = predict(
    big_model_direct, soy_test[ALL_COLUMNS]
)
print(f"Full Mushroom baseline accuracy: "
      f"{accuracy(soy_test[TARGET_MUSHROOM], full_predictions_direct):.2%}")


C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count /

The prob of the featue leaves based on the calss 2-4-d-injury is Zero
The prob of the featue hail based on the calss alternarialeaf-spot is Zero
The prob of the featue leaves based on the calss alternarialeaf-spot is Zero
The prob of the featue leafspots-halo based on the calss alternarialeaf-spot is Zero
The prob of the featue leafspots-marg based on the calss alternarialeaf-spot is Zero
The prob of the featue leafspot-size based on the calss alternarialeaf-spot is Zero
The prob of the featue leaf-malf based on the calss alternarialeaf-spot is Zero
The prob of the featue leaf-mild based on the calss alternarialeaf-spot is Zero
The prob of the featue lodging based on the calss alternarialeaf-spot is Zero
The prob of the featue stem-cankers based on the calss alternarialeaf-spot is Zero
The prob of the featue canker-lesion based on the calss alternarialeaf-spot is Zero
The prob of the featue fruiting-bodies based on the calss alternarialeaf-spot is Zero
The prob of the featue external-d

C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count / len(subset_class))
C:\Users\Amirreza\AppData\Local\Temp\ipykernel_17428\3093499030.py:22: RuntimeWarning: divide by zero encountered in log
  prob = np.log(match_count /

1: از دلایل ممکن برای این واقعه این هست که ممن کهست تعداد کلاس ها به حدی زیاد باشهکه این احتمالی که ممکن هست به هرکدوم از این کلاس ها برسه نزدیک به ضفر دربیاد   
مورد دیگری که هست این امکان وحود داره که به علت ویژگی های زیادی که یک نمونه داده می تونه داشته باشه با ضرب شدن این داده ها  
در هم دیگه باعث کم شدن بیش از حد این آمار بشه   
یک دلیل دیگر هم که ممکن هست باشد این هست که ویژگی در داده های test دیده بشود که قبلا در داده های train نبود و همین باعث شده اختمالی که  
به آن نسبت داده می شود 0 باشه و در صورت ضرب شدن با سایر احتمال ها کل این فرآیند احتمال را 0 بکند  
2: در مشکل Underflow ویژگی های دیده شده در داده های train test مشترک هستند و این 0 شدن احتمال بخاطر توالی موحود در بین ضرب شدن های   
احتمالی هست ولی این مشکل که در قسمت 1 رخ داد به علت نبود برخی از ویژگی هایی هست که در test حضور داشت ولی در train نبود.  
3: وقتی همه این داده های 0 هستند model کلاس به بر اساس خروف الفلا انتخاب می کند به این صورت که a مقدم تر از b می شود(البته این نکته هم هست که   
این روند در دیکشنری هست که داده های در اون ذخیره شدن و روند انتخاب در صورت برابری داده ها به همین صورت هست)

## Part6

## Part 7